![image_1780382811607.png](./image_1780382811607.png "image_1780382811607.png")

![image_1780382839965.png](./image_1780382839965.png "image_1780382839965.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

# Initialize Spark session
spark = SparkSession.builder.appName("DepartmentsExpensesData").getOrCreate()

# Departments dataset
departments_data = [
    (1, "Engineering", 500000),
    (2, "Marketing", 200000),
    (3, "Sales", 300000),
    (4, "HR", 100000)
]

departments_columns = ["dept_id", "dept_name", "budget"]

departments_df = spark.createDataFrame(departments_data, departments_columns)

# Expenses dataset
expenses_data = [
    (1, 1, 150000, "2023-01-15"),
    (2, 1, 200000, "2023-03-20"),
    (3, 1, 180000, "2023-06-10"),
    (4, 2, 250000, "2023-02-28"),
    (5, 3, 100000, "2023-04-05"),
    (6, 3, 150000, "2023-07-22")
]

expenses_columns = ["expense_id", "dept_id", "amount", "expense_date"]

expenses_df = spark.createDataFrame(expenses_data, expenses_columns)

# Show both DataFrames
print("Departments DataFrame:")
departments_df.show()

print("Expenses DataFrame:")
expenses_df.show()


In [0]:
result_df=(
    departments_df.alias("d")
    .join(expenses_df.alias("e"), on="dept_id",how="left")
    .groupBy("dept_name","budget")
    .agg(
        f.coalesce(f.sum("e.amount"),f.lit(0)).alias("total_expenses")
    )
    .select(
        "dept_name",
        "budget",
        "total_expenses",
        (f.col("budget") - f.col("total_expenses")).alias("variance"))
    .orderBy(f.col("variance"))
    )
result_df.show()